# A2.4 · Just-in-time authority

**Function A — Securing AI Architectures → Securing the Architecture — Identity and Ingress**  ·  *Security of AI*

Builds on **[A2.3 · Delegation that narrows, and survives audit](https://spbreed.github.io/cyber-commons/lessons/A2.3.html)**.

| | |
|---|---|
| Tools used | Keycloak, OPA |

## What this lesson is

**What it covers.** Issue a scoped grant, use it, then replay it after expiry and after the task closed.

**Why a security engineer needs it.** Permanent scope makes every injection a successful one, because the authority is always there when the attacker arrives. The control it builds is: short-lived, purpose-bound grants issued per task and expiring with it.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Standing authority means a successful injection always finds a live credential waiting. Just-in-time authority means the attacker has to arrive during the ninety seconds the grant exists, and be doing the one task it was scoped to.

> **At CyberTravels.** Standing payments scope means a successful injection always finds a live refund credential. Just-in-time means the attacker has to arrive during the ninety seconds a specific booking is being settled. R1, R5.

## 2 · The framework

```
   standing                       just-in-time
   +-----------------+            +---------------------+
   | granted once    |            | granted per task    |
   | lives forever   |            | expires in 90s      |
   | any task        |            | this resource only  |
   +-----------------+            +---------------------+
   injection always finds         injection has to arrive
   a live credential              during the window, on that task
```

**Mitigates: T3 Privilege Compromise · T2 Tool Misuse.**

A2.3 narrows authority at the moment of delegation. This lesson removes it when
the task is over.

Standing authority is the reason an injection is always worth attempting: the
credential is there, permanently, waiting. Every successful A1.3 lands on a live
grant. Just-in-time authority changes the arithmetic — the attacker has to
arrive during a window that exists only while a specific task is running, and
that is bound to a specific resource.

Three properties, and the third is the one usually skipped:

**Short-lived.** Minutes, not months. Theft has a deadline.

**Purpose-bound.** Scoped to *this* resource, not to the resource class. Not
`reports:write` but `reports:write` on report 8812.

**Revoked on completion.** The grant ends when the task ends, not when the timer
does. A task that finishes in ten seconds should not leave a fifteen-minute
credential lying around, which is the difference between a TTL and an actual
lifecycle.

The operational cost is real and worth stating plainly: something must issue
these, and if that path breaks, work stops. That is the trade — a system that
fails closed under a control outage, in exchange for a system that has no
standing authority to steal.

> **What this control closes.**
>
> Removes the **standing** grant an injection needs. The attacker must now arrive inside a window bound to one task and one resource.

## 3 · Finding the standing grants, as a skill

Just-in-time authority is only worth building where standing authority exists today, and at CyberTravels that list is not the one in the design document — it is in the authorisation graph and in the OAuth scopes the credential provider stored when someone first connected the payments API. The procedure diffs what each identity *holds* against what its declared tools actually *need*, and flags every permanent grant. This is the file in this repository:

In [ ]:
# skills/attestation/entitlement-overprivilege-analyzer/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: entitlement-overprivilege-analyzer
description: >-
  Analyse entitlement and identity registries to find over-privileged access
  held by a deployment's identities. Use to check granted entitlements
  against what the declared tools actually need, to find over-broad OAuth
  scopes on credential providers, or to find standing privilege.
allowed-tools: Bash, Read
---

# Entitlement Overprivilege Analyzer

**Controls:** Controls 1 and 3 — over-privileged access

## What this adds beyond the IAM verifier

The IAM verifier measures cloud permissions against cloud usage. This skill
measures **application-level entitlements** — relationship-graph authorisation,
OAuth scopes on stored credential providers, and identity-registry
relationships — against what the declared tool surface actually requires.

An agent can hold a minimal cloud role and an OAuth token with full mailbox
access.

## Procedure

1. **Take the required capability set** from the code-surface analyzer. This is
   the denominator: what the declared tools genuinely need.
2. **Enumerate granted entitlements** from the authorisation graph for every
   identity in the deployment manifest.
3. **Enumerate credential-provider scopes.** Stored OAuth scopes are frequently
   far wider than the tool needs, because the consent screen offered a bundle.
4. **Diff.** Every grant with no corresponding requirement is an over-privilege
   finding, and each needs a justification gap recorded — the grant, the
   requirement it was presumably for, and the absence.
5. **Flag standing privilege.** Any grant that is permanent rather than issued
   per task.

## Output contract

```json
{
  "deployment_id": "str",
  "identities": [
    {"id": "str",
     "granted": ["str"],
     "required_by_tools": ["str"],
     "excess": [{"grant": "str", "justification_gap": "str"}]}
  ],
  "oauth_scope_excess": [{"provider": "str", "granted_scope": "str", "needed_scope": "str"}],
  "standing_privilege": ["str"],
  "verdict": "PASS|PARTIAL|FAIL"
}
```

## Failure modes

- **Comparing grants against other grants.** The denominator is the tool
  surface, not a peer deployment.
- **Accepting a bundled OAuth scope** because it was what the provider offered.
- **Missing standing privilege** because the scope itself looked narrow.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/attestation/entitlement-overprivilege-analyzer/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/attestation/entitlement-overprivilege-analyzer/scripts/entitlement_overprivilege_analyzer.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Bind a grant to one scope, one resource and one task, and show what it refuses once the task closes.

This is the executable half of the `entitlement-overprivilege-analyzer` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

GRANTS = {}
CLOCK = {"now": 1000}

def grant(task_id, principal, scope, resource, ttl=120):
    """Purpose-bound: this scope, on this resource, for this task."""
    GRANTS[task_id] = {"principal": principal, "scope": scope, "resource": resource,
                       "expires": CLOCK["now"] + ttl, "open": True}
    return task_id

def use(task_id, scope, resource):
    g = GRANTS.get(task_id)
    if not g:                                return False, "no such grant"
    if not g["open"]:                        return False, "task closed"
    if CLOCK["now"] > g["expires"]:          return False, "expired"
    if scope != g["scope"]:                  return False, f"scoped to {g['scope']}"
    if resource != g["resource"]:            return False, f"bound to {g['resource']}"
    return True, "permitted"

def close(task_id):
    if task_id in GRANTS:
        GRANTS[task_id]["open"] = False       # revoked on completion, not on expiry

grant("t-1", "dana@corp", "reports:write", "report/8812")

attempts = [
 ("the task's own write",        "reports:write", "report/8812"),
 ("a different report",          "reports:write", "report/9999"),
 ("a different scope",           "db:admin",      "report/8812"),
]
for label, scope, resource in attempts:
    ok, why = use("t-1", scope, resource)
    print(f"   {label:26s}{'ok' if ok else 'REFUSED':8s}{why}")

close("t-1")
ok, why = use("t-1", "reports:write", "report/8812")
print(f"   {'after the task completes':26s}{'ok' if ok else 'REFUSED':8s}{why}")

CLOCK["now"] = 2000
GRANTS["t-2"] = dict(GRANTS["t-1"], open=True, expires=1500)
ok, why = use("t-2", "reports:write", "report/8812")
print(f"   {'after the TTL expires':26s}{'ok' if ok else 'REFUSED':8s}{why}")
print()
print("An injection landing at 09:14 needs a task to be open, on the resource")
print("it wants, holding the scope it wants. Standing authority required none")
print("of those three things to line up.")
assert use("t-1", "reports:write", "report/8812")[0] is False

## What you just proved

The skill loads and reports its shape. Note what its procedure insists on: the denominator is the capability set the tools require, not another set of grants — comparing grants against grants is how a review concludes that an over-privileged agent is normal — and a narrow-looking scope still counts as standing privilege if it never expires.

## Your turn

Take one standing grant an agent holds and work out what would break if it expired in two minutes. That list is the real cost of just-in-time, and it is usually shorter than expected.

---

**Next → [A2.5 · The non-human identity lifecycle](https://spbreed.github.io/cyber-commons/lessons/A2.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*